# Requirement 15: Deterministic termination

> Destroy the session environment and discard its secrets on completion or failure.

**Covered by:** Both

**How it is fulfilled:** The orchestrator deletes the sandbox in a `finally` block on both success and failure paths and discards its copy of the key. Deletion removes the sandbox and its disk.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
def run_session(fail):
    sandbox, session_id = new_session()
    key = install_session_key(sandbox, session_id)
    try:
        if fail:
            raise RuntimeError("simulated failure")
        return session_id, "completed"
    except Exception:
        return session_id, "failed"
    finally:
        cleanup(sandbox)
        del key

In [3]:
for fail in (False, True):
    sid, outcome = run_session(fail)
    left = list(orchestrator().list_sandboxes(labels={"session": sid}))
    print(f"session {sid} {outcome}; sandboxes left: {len(left)}")
    check(f"{outcome} path leaves nothing behind", len(left) == 0)

session d1233118 completed; sandboxes left: 0
PASS  completed path leaves nothing behind


session 9381f474 failed; sandboxes left: 0
PASS  failed path leaves nothing behind


**Caveat:** Deletion is confirmed through the API. That the disk contents are unrecoverable is shown only indirectly in requirement 21.